In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns

In [ ]:
df=pd.read_csv(r"E:\Data\train(1).csv")

In [ ]:
df.head

In [ ]:
df.index

In [ ]:
df.shape

In [ ]:
column = df.columns 
for i in column:
    print(df.dtypes[i])

In [ ]:
column=df.columns
for i in column:
    print(f"{i} = {df[i].isnull().sum()}")

In [ ]:
print(df[i].info())

In [ ]:
for i in column:
    if not pd.api.types.is_numeric_dtype(df[i]): 
        print(f"{i}") # Check if column is numeric
        continue  # Skip non-numeric column
    std_ex = df[i].std()
    if (std_ex==0 or std_ex>100):
        print(f"{i} = {std_ex}")


In [ ]:
df.head(5)

In [ ]:
df=df.drop('ID_code',axis=1)
df.head(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns  # Assuming you already imported seaborn

#Calculate the correlation matrix with respect to target
correlation_matrix = df.corr()

# Visualize the correlation matrix with target
plt.figure(figsize=(10, 16))

# Create a heatmap for features correlated with target
sns.heatmap(correlation_matrix[['target']].sort_values(by='target', ascending=False), 
            annot=True, fmt=".2f", cmap='coolwarm', cbar=True, linewidths=0.5, linecolor='black')

# Set title
plt.title('Correlation of Features with target', fontsize=18)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Feature set and target variable
X = df[["var_53", "var_99", "var_133", "var_179", "var_78", "var_94", "var_173", "var_147", "var_95"]]
y = df["target"]

# Split data into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=200, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=50, max_depth=5, learning_rate=0.1, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5)
}

# Train and evaluate models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.4f}")


In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to oversample the minority class
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Train and evaluate models with resampled data
for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} (SMOTE) Accuracy: {accuracy:.4f}")


In [ ]:
from sklearn.feature_selection import RFE

# Example: Apply RFE to Logistic Regression as an example
rfe = RFE(estimator=LogisticRegression(), n_features_to_select=5)
X_train_rfe = rfe.fit_transform(X_train_resampled, y_train_resampled)
X_test_rfe = rfe.transform(X_test_scaled)

# Train models after feature selection
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_rfe, y_train_resampled)
        y_pred = model.predict(X_test_rfe)
        accuracy = accuracy_score(y_test, y_pred)
        print(f"{name} (RFE) Accuracy: {accuracy:.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_halving_search_cv  # Required for HalvingRandomSearchCV
from sklearn.model_selection import HalvingRandomSearchCV

# Sample a smaller portion of the data for RandomizedSearchCV
X_train_sampled, _, y_train_sampled, _ = train_test_split(X_train_resampled, y_train_resampled, test_size=0.8, random_state=42)

# Narrowed parameter grid for faster search
param_grid_rf = {
    'n_estimators': [50, 75],  # Reduced further
    'max_depth': [10, 15],     # Narrower choices
    'min_samples_split': [10], # Single value
    'min_samples_leaf': [2],   # Single value
    'class_weight': ['balanced']
}

# Define RandomForest model
rf_model = RandomForestClassifier(n_jobs=-1, warm_start=True)

# Perform HalvingRandomSearchCV for more efficient search
rf_search = HalvingRandomSearchCV(
    rf_model, 
    param_distributions=param_grid_rf, 
    factor=2,          # Halving factor for efficiency
    random_state=42, 
    n_jobs=-1,         # Utilize all available cores
    verbose=2
)

# Fit the search on a smaller sample
rf_search.fit(X_train_sampled, y_train_sampled)

# Get the best RandomForest model after tuning
best_rf = rf_search.best_estimator_

# Use the best model to make predictions on the full test set
y_pred_rf = best_rf.predict(X_test_scaled)

# Calculate the accuracy score
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest (Tuned with HalvingRandomSearchCV) Accuracy: {accuracy_rf:.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv  # Required for enabling HalvingRandomSearchCV
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import accuracy_score

# Narrowed parameter grid for faster search
param_grid_rf = {
    'n_estimators': [50, 100],        # Reduced number of trees
    'max_depth': [10, 20],            # Narrowed depth options
    'min_samples_split': [5, 10],     # Fewer split choices
    'min_samples_leaf': [1, 2],       # Fewer leaf node choices
    'class_weight': ['balanced']      # Addressing class imbalance
}

# Define RandomForest model
rf_model = RandomForestClassifier(n_jobs=-1, warm_start=True)

# Perform HalvingRandomSearchCV for more efficient search
rf_search = HalvingRandomSearchCV(
    rf_model, 
    param_distributions=param_grid_rf, 
    factor=2,          # Halving factor
    random_state=42, 
    n_jobs=-1,         # Utilize all available cores
    verbose=2
)

# Fit the model using the HalvingRandomSearchCV
rf_search.fit(X_train_resampled, y_train_resampled)

# Get the best Random Forest model after tuning
best_rf = rf_search.best_estimator_

# Predict on the test set
y_pred_rf = best_rf.predict(X_test_scaled)

# Check the accuracy of the tuned model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest (Tuned with HalvingRandomSearchCV) Accuracy: {accuracy_rf:.4f}")
